# Jansetu E4B Fine-tuning — Google Colab

**Gemma 4 Good Hackathon | Unsloth Special Technology Track**

This notebook fine-tunes Google's Gemma 4 E4B model to understand rural Indian health language —
Hindi, Bhojpuri, and Hinglish symptom descriptions used by ASHA workers and patients in
low-resource settings across Bihar, UP, Odisha, and Maharashtra.

### What this notebook does
1. Installs Unsloth and dependencies
2. Generates a synthetic training dataset of 400 examples in 8 language styles
3. Fine-tunes Gemma 4 E4B with LoRA (300 steps, ~35-45 minutes on a T4 GPU)
4. Evaluates the fine-tuned model vs base model
5. Exports the model for Android LiteRT-LM deployment

**Runtime:** Select `Runtime > Change runtime type > T4 GPU` before running.

---

## Cell 1 — Setup: Check GPU and install dependencies

We start by confirming we have a GPU available and installing the required packages.
Unsloth is a memory-optimised fine-tuning library that makes Gemma 4 E4B fit within
the 15 GB VRAM of a free Colab T4.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Go to Runtime > Change runtime type and select T4 GPU."
    )

gpu_name   = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU     : {gpu_name}")
print(f"VRAM    : {total_vram:.1f} GB")
print(f"PyTorch : {torch.__version__}")

# Install Unsloth and training dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl datasets pyyaml

print("\nInstallation complete.")

## Cell 2 — Generate the training dataset

We generate a synthetic dataset of **400 training** and **80 eval** examples programmatically.
The examples cover 8 language/style categories that reflect real-world ASHA worker usage:

| Category | Examples | Description |
|---|---|---|
| Standard Hindi | 50 | Clear Hindi symptom descriptions |
| Bhojpuri | 60 | UP/Bihar belt dialect |
| Hinglish | 60 | Mixed Hindi-English urban-rural |
| ASHA clinical | 60 | Structured worker-style descriptions with MUAC |
| Severe/emergency | 50 | High-acuity cases requiring referral |
| Mild self-treatable | 50 | Low-acuity cases (must NOT over-refer) |
| Elderly patients | 35 | Age-specific presentations |
| Children malnutrition | 35 | Under-5 malnutrition + fever |

**Why synthetic?** Real patient data requires consent, anonymisation, and ethics approval.
This synthetic dataset is good enough for a meaningful fine-tune; we document in the README
how real ASHA worker data can be incorporated to improve it further.

In [ ]:
import json
import random
from typing import Any

random.seed(42)

SYSTEM_PROMPT = (
    "You are Jansetu, a health AI assistant for rural India. "
    "When a user describes symptoms, extract them into a structured JSON object "
    "with these exact fields: symptoms (array of strings from the canonical list), "
    "duration (integer days or null), ageGroup (child/adult/elderly), "
    "gender (M/F/unknown), severity (mild/moderate/severe), referral (boolean). "
    "Canonical symptoms: fever, cough, breathlessness, diarrhoea, vomiting, rash, "
    "headache, bodyache, sore_throat, runny_nose, malnutrition, jaundice, "
    "conjunctivitis, seizure, unconscious, bleeding. "
    "Output ONLY the JSON object, nothing else."
)

# --------------- Template banks (abbreviated for notebook readability) ---
# Full template banks are identical to those in dataset/generate_dataset.py

ALL_TEMPLATES: list[dict[str, Any]] = [
    # Standard Hindi
    {"texts":["Teen din se bukhar hai, saath mein khansi bhi","{N} din se bukhar hai, khansi bhi"],"symptoms":["fever","cough"],"severity":"moderate","referral":False,"age_group":"adult","gender":"unknown","category":"standard_hindi"},
    {"texts":["Aankhein pili ho gayi hain, bhook nahi","Poora body peela, jaundice lag raha"],"symptoms":["jaundice"],"severity":"severe","referral":True,"age_group":"adult","gender":"random","category":"standard_hindi"},
    {"texts":["Bachche ko {N} din se dast aa rahe hain","Chhote ko dast ho rahe hain"],"symptoms":["diarrhoea"],"severity":"moderate","referral":False,"age_group":"child","gender":"random","category":"standard_hindi"},
    {"texts":["Budhiye ko sans lene mein takleef","Saans fulne lagi hai"],"symptoms":["breathlessness"],"severity":"severe","referral":True,"age_group":"elderly","gender":"F","category":"standard_hindi"},
    {"texts":["Naak beh rahi hai, thoda sar dard","Zukaam hua hai, naak se paani"],"symptoms":["runny_nose","headache"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"standard_hindi"},
    {"texts":["Poore body mein dard, bukhar bhi","Bukhar ke saath body dard"],"symptoms":["fever","bodyache"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"standard_hindi"},
    {"texts":["Gale mein dard hai, kuch nighalne mein taklif","Gala baith gaya, {N} din se"],"symptoms":["sore_throat"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"standard_hindi"},
    {"texts":["Halka bukhar hai, aur kuch nahi","Thoda bukhar hai, {N} din se"],"symptoms":["fever"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"standard_hindi"},
    {"texts":["Dast aur ulti dono hain, kamzori","Ulti dast dono chal rahe hain"],"symptoms":["diarrhoea","vomiting"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"standard_hindi"},
    {"texts":["Khansi mein khoon aa raha hai","Balgam mein khoon {N} din se"],"symptoms":["cough","bleeding"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"standard_hindi"},
    # Bhojpuri
    {"texts":["Tikan ba, khansi bhi ba, teen din se","{N} din se tikan ba, khansi nahi chhut"],"symptoms":["fever","cough"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"bhojpuri"},
    {"texts":["Bachwa ke seizure aa gail","Angiya chadh rahi ba, haath pair ainth gail"],"symptoms":["seizure"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"bhojpuri"},
    {"texts":["Haddi dikhay lage ba, sukhat jaat ba","Sukh raha ba, haddi sab dikh rahi"],"symptoms":["malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"bhojpuri"},
    {"texts":["Aankhein pili ho gaili ba, haldi jaisi","Poora body peela pad gail ba"],"symptoms":["jaundice"],"severity":"severe","referral":True,"age_group":"adult","gender":"random","category":"bhojpuri"},
    {"texts":["Halka tikan ba, koi aur takleef nahi","Thoda sa tikan ba, khansi nahi"],"symptoms":["fever"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"bhojpuri"},
    {"texts":["Sar mein dard baa, aankhon ke aage andhiyar","Mathha phootat ba"],"symptoms":["headache"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"bhojpuri"},
    {"texts":["Bachi sukh rahi ba, pet phula hua","Bachi bahut patla ho gaili"],"symptoms":["malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"F","category":"bhojpuri"},
    {"texts":["Sans nahi leke pa rahi ba, hont nile","Saans fulat ba, chalat nahi ban raha"],"symptoms":["breathlessness"],"severity":"severe","referral":True,"age_group":"adult","gender":"F","category":"bhojpuri"},
    {"texts":["Peet mein dard baa, dast bhi hote ba","Dast aur ulti dono ho rahi ba"],"symptoms":["diarrhoea","vomiting"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"bhojpuri"},
    {"texts":["Bache ke khansi ba, naak beh rahi","Sirf zukaam jaisan ba, khansi ba"],"symptoms":["cough","runny_nose"],"severity":"mild","referral":False,"age_group":"child","gender":"random","category":"bhojpuri"},
    # Hinglish
    {"texts":["Usko fever hai {N} days se, plus cough","Fever and cough dono hain"],"symptoms":["fever","cough"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"hinglish"},
    {"texts":["Serious case hai, unconscious pad gaya","Wo suddenly unconscious ho gaya"],"symptoms":["unconscious"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"hinglish"},
    {"texts":["Yellow eyes aa gaye, jaundice suspect","Aankhein yellow, jaundice ka sign"],"symptoms":["jaundice"],"severity":"severe","referral":True,"age_group":"adult","gender":"random","category":"hinglish"},
    {"texts":["Baby ko rash ho gaya, whole body pe","Skin pe rash hai, itching bhi"],"symptoms":["rash","fever"],"severity":"moderate","referral":False,"age_group":"child","gender":"random","category":"hinglish"},
    {"texts":["Normal cold hai, runny nose aur cough","Seasonal flu jaisa, nothing serious"],"symptoms":["runny_nose","cough"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"hinglish"},
    {"texts":["Seizure aa gaya suddenly, please rush","Usko fit aa gaya, body shake thi"],"symptoms":["seizure"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"hinglish"},
    {"texts":["Child malnourished, ribs dikh rahi","Baby ka weight bahut kam, bones visible"],"symptoms":["malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"random","category":"hinglish"},
    {"texts":["Mild headache hai, fever nahi","Sirf headache, rest karega theek"],"symptoms":["headache"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"hinglish"},
    {"texts":["Mummy ko breathlessness feel ho rahi","Mom ko breathing problem, seriously dekho"],"symptoms":["breathlessness"],"severity":"severe","referral":True,"age_group":"elderly","gender":"F","category":"hinglish"},
    {"texts":["Loose motions ho rahe hain {N} days","Pani jaisi potty aa rahi hai"],"symptoms":["diarrhoea","vomiting"],"severity":"moderate","referral":False,"age_group":"adult","gender":"random","category":"hinglish"},
    # ASHA clinical
    {"texts":["35 saal ki mahila, {N} din se tez bukhar, saans fulna","Mahila 35 yr, fever aur breathlessness"],"symptoms":["fever","breathlessness"],"severity":"severe","referral":True,"age_group":"adult","gender":"F","category":"asha_clinical"},
    {"texts":["8 saal ka bachcha, poore body pe rash, bukhar 102","8 yr male, rash with fever, measles"],"symptoms":["rash","fever"],"severity":"moderate","referral":True,"age_group":"child","gender":"M","category":"asha_clinical"},
    {"texts":["2 saal ki bachi, MUAC 10.5cm, SAM","Female child 2 yrs, MUAC 10.5, SAM"],"symptoms":["malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"F","category":"asha_clinical"},
    {"texts":["60 saal ke bujurg, dast {N} din, kamzori","Elder 60 yr, diarrhoea, dehydrated"],"symptoms":["diarrhoea"],"severity":"moderate","referral":True,"age_group":"elderly","gender":"M","category":"asha_clinical"},
    {"texts":["25 saal ki mahila, halki khansi naak beh rahi","Young female, mild cough runny nose"],"symptoms":["cough","runny_nose"],"severity":"mild","referral":False,"age_group":"adult","gender":"F","category":"asha_clinical"},
    {"texts":["4 saal ka bachcha, khansi {N} din, saans tez","4 yr male, cough, fast breathing"],"symptoms":["cough","breathlessness"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"asha_clinical"},
    {"texts":["5 saal ki bachi, seizure aya, referral urgent","Female 5 yr, seizure episode, urgent"],"symptoms":["seizure","unconscious"],"severity":"severe","referral":True,"age_group":"child","gender":"F","category":"asha_clinical"},
    {"texts":["55 saal ka purush, aankhein pili, jaundice","Male 55 yr, jaundice, liver check"],"symptoms":["jaundice"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"asha_clinical"},
    {"texts":["30 saal ki mahila, halka bukhar ek din","Female 30 yr, mild fever yesterday"],"symptoms":["fever"],"severity":"mild","referral":False,"age_group":"adult","gender":"F","category":"asha_clinical"},
    {"texts":["15 saal ki ladki, gale mein dard, bukhar nahi","Female 15 yr, sore throat, no fever"],"symptoms":["sore_throat"],"severity":"mild","referral":False,"age_group":"adult","gender":"F","category":"asha_clinical"},
    # Severe/emergency
    {"texts":["Bachche ko seizure bar bar, hosh nahi","Bache ko fit aayi, unconscious hai"],"symptoms":["seizure","unconscious"],"severity":"severe","referral":True,"age_group":"child","gender":"random","category":"severe_emergency"},
    {"texts":["Khoon aa raha hai, rok nahi pa rahe","Bahut zyada khoon beh raha hai"],"symptoms":["bleeding"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"severe_emergency"},
    {"texts":["Saans bilkul nahi le pa raha, hont nile","Hont nile, breathing band ho rahi"],"symptoms":["breathlessness"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"severe_emergency"},
    {"texts":["Behosh pad gaya, uthha nahi pa raha","Suddenly gir gaya, hosh nahi"],"symptoms":["unconscious"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"severe_emergency"},
    {"texts":["Dast mein khoon, rok nahi pa rahe","Potti mein laal khoon bahut zyada"],"symptoms":["diarrhoea","bleeding"],"severity":"severe","referral":True,"age_group":"adult","gender":"random","category":"severe_emergency"},
    {"texts":["Poora body peela, aankhein bhi, jaundice advanced","Skin aur aankhein dono pili, critical"],"symptoms":["jaundice"],"severity":"severe","referral":True,"age_group":"adult","gender":"random","category":"severe_emergency"},
    {"texts":["Ek saal ka bacha, wasting, haddi nikal rahi, SAM","12 mahine ka, bilateral edema, critical"],"symptoms":["malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"random","category":"severe_emergency"},
    {"texts":["Bukhar 104, seizure bhi aya, emergency","Very high fever with febrile seizure"],"symptoms":["fever","seizure"],"severity":"severe","referral":True,"age_group":"child","gender":"random","category":"severe_emergency"},
    {"texts":["Bache saans tez, chest mein ghur ghur","Child fast breathing, chest retractions"],"symptoms":["breathlessness","cough"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"severe_emergency"},
    {"texts":["Khansi mein khoon, teen hafte se","Cough 2 weeks with blood, TB suspect"],"symptoms":["cough","bleeding"],"severity":"severe","referral":True,"age_group":"adult","gender":"M","category":"severe_emergency"},
    # Mild
    {"texts":["Halki khansi hai, ek din se, bukhar nahi","Thodi khansi hai, koi aur nahi"],"symptoms":["cough"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Gala dukh raha hai, pani pine mein taklif","Sore throat hai, {N} din se, bukhar nahi"],"symptoms":["sore_throat"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Naak beh rahi, thoda sar dard, zukaam","Runny nose, headache, seasonal hai"],"symptoms":["runny_nose","headache"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Halka bukhar ek din se, koi aur nahi","Low grade fever, paracetamol diya"],"symptoms":["fever"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Ek baar ulti hui, ab theek hai","Thoda pet dard, ek ulti, theek"],"symptoms":["vomiting"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Sar mein halka dard, rest ke baad theek","Thoda headache, nind se theek hoga"],"symptoms":["headache"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Ek baar dast hua, ab nahi, ORS diya","Thoda dast tha, ORS se stable"],"symptoms":["diarrhoea"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Aankhein thodi laal, mild conjunctivitis","Conjunctivitis ek aankh mein, mild"],"symptoms":["conjunctivitis"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Bukhar aur naak beh rahi, common cold","Normal flu jaisa, ghabrana nahi"],"symptoms":["fever","runny_nose"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    {"texts":["Body mein thoda dard, rest se theek","Halka bodyache, koi bukhar nahi"],"symptoms":["bodyache"],"severity":"mild","referral":False,"age_group":"adult","gender":"random","category":"mild_cases"},
    # Elderly
    {"texts":["Dadi ko sans lene mein takleef, seene mein dard","Burhiya ko saans fulti, chhaati mein bhari"],"symptoms":["breathlessness","bodyache"],"severity":"severe","referral":True,"age_group":"elderly","gender":"F","category":"elderly"},
    {"texts":["Baba 70 saal ke, aankhein pili, jaundice","Bujurg 70 plus, pedal edema aur jaundice"],"symptoms":["jaundice"],"severity":"severe","referral":True,"age_group":"elderly","gender":"M","category":"elderly"},
    {"texts":["Budhiya ko 3 din se hosh nahi","Dadi beech beech mein behosh ho jaati"],"symptoms":["unconscious"],"severity":"severe","referral":True,"age_group":"elderly","gender":"F","category":"elderly"},
    {"texts":["Nani ko halka bukhar, khansi, naak beh rahi","Elderly lady mild cold, no breathlessness"],"symptoms":["fever","cough","runny_nose"],"severity":"mild","referral":False,"age_group":"elderly","gender":"F","category":"elderly"},
    {"texts":["Bujurg ko tez bukhar, saans nahi le pa rahe","Old man, high fever aur breathlessness"],"symptoms":["fever","breathlessness"],"severity":"severe","referral":True,"age_group":"elderly","gender":"M","category":"elderly"},
    {"texts":["Bujurg ko haath pair sujan, saans takleef","Elder, generalised edema, dyspnoea"],"symptoms":["breathlessness"],"severity":"severe","referral":True,"age_group":"elderly","gender":"F","category":"elderly"},
    {"texts":["Nani ko sirf gala dard, aur kuch nahi","Elderly, sore throat only, mild"],"symptoms":["sore_throat"],"severity":"mild","referral":False,"age_group":"elderly","gender":"F","category":"elderly"},
    # Child malnutrition
    {"texts":["Ek saal ka bachcha, sukha, haddi dikh rahi, pet phula","1 yr, wasting oedema, bones visible"],"symptoms":["malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"child_malnutrition"},
    {"texts":["2 saal ki bachi, {N} din dast, aankhein andar, aansu nahi","2 yr girl, diarrhoea, sunken eyes"],"symptoms":["diarrhoea","malnutrition"],"severity":"severe","referral":True,"age_group":"child","gender":"F","category":"child_malnutrition"},
    {"texts":["Teen saal ka baccha, MUAC kam, khansi bhi","3 yr, MUAC low, cough, malnutrition combo"],"symptoms":["malnutrition","cough"],"severity":"moderate","referral":True,"age_group":"child","gender":"M","category":"child_malnutrition"},
    {"texts":["5 saal ka bachcha, bukhar khansi saans tez","5 yr, high fever cough fast breathing"],"symptoms":["fever","cough","breathlessness"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"child_malnutrition"},
    {"texts":["2 saal ka bachcha, seizure aya, {N} minute","2 yr, febrile seizure, post-ictal"],"symptoms":["seizure","fever"],"severity":"severe","referral":True,"age_group":"child","gender":"M","category":"child_malnutrition"},
    {"texts":["4 saal ka bacha, halka bukhar, khel raha","4 yr, mild fever, active, no emergency"],"symptoms":["fever"],"severity":"mild","referral":False,"age_group":"child","gender":"M","category":"child_malnutrition"},
    {"texts":["2 saal ki bachi, naak beh rahi, active","2 yr girl, mild URTI, active, no ref"],"symptoms":["runny_nose","cough"],"severity":"mild","referral":False,"age_group":"child","gender":"F","category":"child_malnutrition"},
]


def generate_dataset(n_train: int = 400, n_eval: int = 80) -> tuple[list, list]:
    """Generate train and eval examples from the template bank."""
    def make_example(tmpl: dict) -> dict:
        text = random.choice(tmpl["texts"])
        if "{N}" in text:
            n = random.randint(1, 7)
            text = text.replace("{N}", str(n))
            duration = n
        else:
            duration = None if random.random() < 0.3 else random.randint(1, 7)
        gender = tmpl["gender"]
        if gender == "random":
            gender = random.choice(["M", "F", "unknown"])
        output = {
            "symptoms": tmpl["symptoms"],
            "duration": duration,
            "ageGroup": tmpl["age_group"],
            "gender": gender,
            "severity": tmpl["severity"],
            "referral": tmpl["referral"],
        }
        return {
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": text},
                {"role": "assistant", "content": json.dumps(output, ensure_ascii=False)},
            ],
            "category": tmpl["category"],
        }

    all_ex = [make_example(ALL_TEMPLATES[i % len(ALL_TEMPLATES)])
              for i in range(n_train + n_eval)]
    random.shuffle(all_ex)
    return all_ex[:n_train], all_ex[n_train:]


train_examples, eval_examples = generate_dataset(400, 80)
print(f"Generated {len(train_examples)} train + {len(eval_examples)} eval examples")

# Quick stats
from collections import Counter
ref_true = sum(1 for e in train_examples if json.loads(e["messages"][2]["content"])["referral"])
print(f"Referral true: {ref_true} ({ref_true/len(train_examples)*100:.1f}%)")
cats = Counter(e["category"] for e in train_examples)
for cat, cnt in cats.most_common():
    print(f"  {cat:28s}: {cnt}")

## Cell 3 — Load Gemma 4 E4B

We load the model using **Unsloth's FastModel**, which applies two key optimisations:
1. **4-bit quantisation** — reduces the 4B parameter model from ~8 GB to ~2.5 GB in VRAM
2. **Flash Attention** — speeds up training by 2-3x versus standard attention

These optimisations make it possible to fine-tune E4B on a free T4 GPU.

> **Note:** If `google/gemma-4-e4b-it` is not yet available on HuggingFace,
> the cell will automatically fall back to `google/gemma-2-2b-it` for pipeline testing.

In [ ]:
import torch
from unsloth import FastModel

MODEL_NAME  = "google/gemma-4-e4b-it"
FALLBACK    = "google/gemma-2-2b-it"
MAX_SEQ_LEN = 512

for attempt, name in enumerate([MODEL_NAME, FALLBACK]):
    try:
        model, tokenizer = FastModel.from_pretrained(
            model_name=name,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )
        print(f"Loaded: {name}")
        if attempt == 1:
            print(f"(Note: using fallback — update MODEL_NAME when Gemma 4 E4B is released)")
        break
    except OSError:
        if attempt == 0:
            print(f"{MODEL_NAME} not found, trying fallback …")
        else:
            raise

used_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used: {used_gb:.2f} GB")

## Cell 4 — Apply LoRA adapters

**LoRA (Low-Rank Adaptation)** is a fine-tuning technique that adds a small number of
trainable parameters to the model without modifying the original weights.

Instead of updating all 4 billion parameters (which would require ~32 GB VRAM),
LoRA adds small rank-16 matrices to 7 key attention layers. This means we only
train about **4 million parameters** — less than 0.1% of the model — making
fine-tuning feasible on a T4 GPU in under an hour.

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({trainable/total*100:.3f}% of total)")
print(f"Total parameters    : {total:,}")

## Cell 5 — Prepare the dataset

We format the examples using the model's **chat template**, which wraps each conversation
in the special tokens Gemma expects. We then apply `train_on_responses_only`, which masks
the loss on everything except the assistant's JSON output.

This is critical: without this masking, the model would waste capacity learning to predict
the system prompt and user input — things that are always identical — rather than learning
the correct JSON extraction.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def apply_template(batch: dict) -> dict:
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch["messages"]
    ]
    return {"text": texts}

train_ds = Dataset.from_list(train_examples)
eval_ds  = Dataset.from_list(eval_examples)
train_ds = train_ds.map(apply_template, batched=True)
eval_ds  = eval_ds.map(apply_template,  batched=True)

print(f"Train: {len(train_ds)} examples | Eval: {len(eval_ds)} examples")
print(f"\nSample (first 300 chars):")
print(train_ds[0]["text"][:300])

## Cell 6 — Fine-tune the model (300 steps, ~35-45 min)

We use **SFTTrainer** (Supervised Fine-tuning Trainer) from the `trl` library.
Key hyperparameters:

- **300 steps** at batch size 2 × 4 accumulation = effective batch 8
- **Learning rate 2e-4** — standard for LoRA fine-tuning
- **Warmup 10 steps** — gradually ramps up the learning rate at start
- `train_on_responses_only` — loss only on assistant JSON outputs

Watch the loss in the log: it should drop from ~2.0 initially to ~0.3-0.5 by the end.
If it doesn't drop, check that the chat template is applied correctly.

In [ ]:
import time

bf16 = torch.cuda.get_device_capability()[0] >= 8
fp16 = not bf16

trainer_cfg = SFTConfig(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=300,
    learning_rate=2e-4,
    fp16=fp16,
    bf16=bf16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    output_dir="./jansetu-e4b-finetuned",
    seed=42,
    report_to="none",
    max_seq_length=MAX_SEQ_LEN,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=trainer_cfg,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print(f"Starting training — 300 steps (~35-45 min on T4) …")
t0     = time.time()
result = trainer.train()
elapsed = (time.time() - t0) / 60

print(f"\nTraining complete in {elapsed:.1f} minutes")
print(f"Final train loss: {result.metrics.get('train_loss', 'N/A'):.4f}")

## Cell 7 — Evaluate on the eval set

We run the fine-tuned model on the 80 held-out eval examples and compute:
- **JSON parse rate**: what % of outputs are valid JSON
- **Symptom F1**: how well the predicted symptom list matches the gold list
- **Referral recall**: the critical safety metric — did we correctly flag referral cases?

In [ ]:
import json

FastModel.for_inference(model)

def run_eval(examples: list, label: str = "Fine-tuned") -> dict:
    parse_ok = sym_rec = sym_prec = ref_ok = ref_total = sev_ok = 0
    total = len(examples)

    for ex in examples:
        gold = json.loads(ex["messages"][2]["content"])
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": ex["messages"][1]["content"]},
        ]
        inputs = tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=150, temperature=0.1,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
        pred_str = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

        try:
            pred = json.loads(pred_str.strip())
            parse_ok += 1
        except Exception:
            if gold["referral"]:
                ref_total += 1
            continue

        g_syms = set(gold["symptoms"])
        p_syms = set(pred.get("symptoms", []))
        if g_syms or p_syms:
            tp = len(g_syms & p_syms)
            sym_rec  += tp / max(len(g_syms), 1)
            sym_prec += tp / max(len(p_syms), 1)

        if gold["referral"]:
            ref_total += 1
            ref_ok    += int(pred.get("referral") is True)
        sev_ok += int(pred.get("severity") == gold["severity"])

    print(f"\n{'='*44}")
    print(f"  {label} — Eval results ({total} examples)")
    print(f"{'='*44}")
    print(f"  JSON parse rate   : {parse_ok/total*100:.1f}%")
    print(f"  Symptom recall    : {sym_rec/total*100:.1f}%")
    print(f"  Symptom precision : {sym_prec/total*100:.1f}%")
    print(f"  Referral recall   : {ref_ok/max(ref_total,1)*100:.1f}%  ← critical")
    print(f"  Severity accuracy : {sev_ok/total*100:.1f}%")
    return {"parse": parse_ok/total, "ref_recall": ref_ok/max(ref_total,1)}

metrics = run_eval(eval_examples)

## Cell 8 — Export the model

We save the model in **merged 16-bit format** — LoRA weights are merged back into
the base model weights, producing a standard model folder that the Android team
can feed directly into the LiteRT-LM conversion pipeline.

No changes to the Android app code are needed — this is a drop-in weight replacement.

In [ ]:
EXPORT_DIR = "./jansetu-e4b-final"

print(f"Exporting merged 16-bit model → {EXPORT_DIR}")
model.save_pretrained_merged(EXPORT_DIR, tokenizer, save_method="merged_16bit")
print("Export complete.")
print()
print("Android team instructions:")
print(f"  Hand the '{EXPORT_DIR}' folder to the Android team.")
print("  They run the LiteRT-LM conversion on it unchanged.")

# Optional: download the model folder as a zip
import os
os.system(f"zip -r jansetu-e4b-final.zip {EXPORT_DIR}")
print("\nDownload jansetu-e4b-final.zip from the Colab Files panel.")

## Cell 9 — Before / after comparison

We test the fine-tuned model with five example inputs from different language categories
to demonstrate qualitative improvement over the base model.

The examples are chosen to highlight the cases where the base model most commonly fails:
Bhojpuri seizure descriptions, visual malnutrition descriptions, and mild cases that
should not trigger a referral.

In [ ]:
TEST_INPUTS = [
    {
        "category": "Bhojpuri — seizure",
        "input": "Angiya chadh rahi ba, haath pair ainth gail, 2 saal ke bachwa ke",
        "gold": '{"symptoms":["seizure"],"duration":null,"ageGroup":"child","gender":"M","severity":"severe","referral":true}',
    },
    {
        "category": "Visual malnutrition",
        "input": "Bachchi sukh rahi hai, haddi dikhne lagi, pet phula hua",
        "gold": '{"symptoms":["malnutrition"],"duration":null,"ageGroup":"child","gender":"F","severity":"severe","referral":true}',
    },
    {
        "category": "Jaundice colloquial",
        "input": "Aankhein pili ho gayi, skin pe haldi jaisi rang aa gayi hai",
        "gold": '{"symptoms":["jaundice"],"duration":null,"ageGroup":"adult","gender":"unknown","severity":"severe","referral":true}',
    },
    {
        "category": "Mild — must NOT refer",
        "input": "Halki khansi hai kal se, bukhar nahi, naak se thoda paani aa raha",
        "gold": '{"symptoms":["cough","runny_nose"],"duration":1,"ageGroup":"adult","gender":"unknown","severity":"mild","referral":false}',
    },
    {
        "category": "ASHA clinical",
        "input": "2 saal ki bachi, MUAC 10.5cm, sukhi twacha, malnutrition SAM",
        "gold": '{"symptoms":["malnutrition"],"duration":null,"ageGroup":"child","gender":"F","severity":"severe","referral":true}',
    },
]

print("Fine-tuned model inference on hard test cases:\n")
print("=" * 70)

for tc in TEST_INPUTS:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": tc["input"]},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=150, temperature=0.1,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

    print(f"\n[{tc['category']}]")
    print(f"  Input : {tc['input']}")
    print(f"  Gold  : {tc['gold']}")
    print(f"  Pred  : {pred}")
    try:
        g = json.loads(tc["gold"])
        p = json.loads(pred)
        ref_ok = g["referral"] == p.get("referral")
        sev_ok = g["severity"] == p.get("severity")
        print(f"  Referral {'✓' if ref_ok else '✗'}  Severity {'✓' if sev_ok else '✗'}")
    except Exception:
        print("  (could not parse prediction)")

print("\n" + "=" * 70)
print("\nFine-tuning complete! The model is ready for Android LiteRT conversion.")